# **A/B Testing E-commerce**

## Context

E-commerce platforms are often using recommendation systems of products with the objective to show itens more relevants to their users. It is expected that these recommendations will increase costumer engagement, and consequently, improve the conversion rate and the site's overall sales volume.

Before introducing a new functionality for all users, it is necessary to evaluate if the change will truly perform better than the old platform version.

A widely approach used to evaluate these changes is the A/B Testing or Split Testing.

[Click here to acess the dataset](https://www.kaggle.com/datasets/zhangluyuan/ab-testing)

### • Importing and loading data

In [1]:
# Importing the libraries that will be used.
import pandas as pd
import numpy as np
import plotly.express as px
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# Loading the dataframe
df = pd.read_csv("/kaggle/input/datasets/zhangluyuan/ab-testing/ab_data.csv")

# Checking if the dataframe was loaded.
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


### • Initial Exploratory Data Analysis (EDA)

In [3]:
# Getting the info about the dataframe.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294478 non-null  int64 
 1   timestamp     294478 non-null  object
 2   group         294478 non-null  object
 3   landing_page  294478 non-null  object
 4   converted     294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


In [4]:
# Converting the timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [5]:
df.nunique()

user_id         290584
timestamp       294478
group                2
landing_page         2
converted            2
dtype: int64

In [6]:
# Checking for duplicate users/rows
df["user_id"].duplicated().sum()

np.int64(3894)

There are 3,894 duplicate records (294,478 - 290,584); these are not necessarily 3,894 duplicate users, but rather duplicate rows in the dataset.

In [7]:
# Checking users who appear more than once
df[df["user_id"].duplicated(keep=False)].sort_values("user_id")

,user_id,timestamp,group,landing_page,converted
213114,630052,2017-01-07 12:25:54.089486,treatment,old_page,1
230259,630052,2017-01-17 01:16:05.208766,treatment,new_page,0
251762,630126,2017-01-19 17:16:00.280440,treatment,new_page,0
22513,630126,2017-01-14 13:35:54.778695,treatment,old_page,0
11792,630137,2017-01-22 14:59:22.051308,control,new_page,0
...,...,...,...,...,...
99479,945703,2017-01-18 06:39:31.294688,control,old_page,0
186960,945797,2017-01-13 17:23:21.750962,control,old_page,0
40370,945797,2017-01-11 03:04:49.433736,control,new_page,1
165143,945971,2017-01-16 10:09:18.383183,control,old_page,0


### • Data Treatment
Objective:
- Identify duplicate records.
- Remove duplicate records.
- Check consistency between`group` and `landing_page`

In [8]:
# Removing duplicate records.
df = df.drop_duplicates(subset="user_id")

In [9]:
# Identifying duplicate records
df["user_id"].duplicated().sum()

np.int64(0)

In [10]:
df[df["user_id"].duplicated(keep=False)]

,user_id,timestamp,group,landing_page,converted


In [11]:
# Checking consistency between`group` and `landing_page`
pd.crosstab(df["group"], df["landing_page"])

landing_page,new_page,old_page
group,,
control,1006,144226
treatment,144314,1038


The `group`column indicates which experiment the user recieved (`control` or `treatment`).

The control group represents the old version of the site (without the recommendation system). And the treatment group represents the new version of the site (with the recommendation system).

Analysing the table above, we can see that is something wrong, because for the experiment to work the control group should have users just in the `old_page` landing page but there are 1,006 users that accessed the `new_page` during the experiment. Also there are some users from treatment group (1038) associated with the `old_page`.

Briefly:
- control group should only have old_page
- treatment group should only have new_page.

For the A/B testing work it is important for the data be distributed properly. However in the current state of the data, it is clear that the experiment was contaminated.

In [12]:
# Let's remove these records to leave only the correct cases.
df = df[
    ((df["group"] == "control") & (df["landing_page"] == "old_page")) |
    ((df["group"] == "treatment") & (df["landing_page"] == "new_page"))
]

In [13]:
# Let's double-check the groups.
pd.crosstab(df["group"], df["landing_page"])

landing_page,new_page,old_page
group,,
control,0,144226
treatment,144314,0


In [14]:
# Reorganizing the dataframe indexes.
df = df.reset_index(drop=True)

### • Analysis with 5000 samples

In [15]:
# Creating a sample of 5000
sample_df = df.sample(n=5000, random_state=42)

In [16]:
# Distribution of users by group
fig = px.pie(
    sample_df,
    names="group",
    title="Control vs. Treatment Ratio (Sample of 5000)"
)

fig.update_traces(
    textinfo="label+percent",
    texttemplate="%{label}<br>%{percent:.2%}"  # Controls decimal places
)

fig.show()

### • Experiment Success Metric

We are evaluating whether the new version of the site, which features a recommendation system, increases the probability of a user making a purchase.

Therefore, the chosen metric is the conversion rate, defined as the proportion of users who completed a purchase after visiting the page.

The conversion rate can be calculated as:

$$Conversion\ Rate = \frac{Number\ of\ users\ who\ purchased}{Total\ number\ of\ users}$$

In the dataset, this metric can be obtained by calculating the mean of the `converted` variable, since it assumes the following values:

- 1 → The user made a purchase
- 0 → The user did not make a purchase

In [17]:
conversion_rates = sample_df.groupby("group")["converted"].mean().reset_index()

fig = px.bar(
    conversion_rates,
    x="group",
    y="converted",
    color="group",
    title="Conversion Rate per Group",
    text="converted"
)

fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')

fig.show()

The graph above presents the average conversion rate for each experimental group.

-`control:` users exposed to the original version of the site (without the recommendation system).
-`treatment:` users exposed to the new version of the site (with the recommendation system).

This visualization allows for an intuitive observation of whether there is any performance difference between the two versions.

We can observe that the new page was not more efficient in generating sales than the old version. In fact, we can see a slightly higher conversion rate in the `control` group.

In addiction to compartin average conversion rates, it is also crucial to analyze how the experiment behaves over time.

### • Conversion over time

In [18]:
sample_df["date"] = sample_df["timestamp"].dt.date

daily_conversion = (
    sample_df.groupby(["date","group"])["converted"]
    .mean()
    .reset_index()
)

fig = px.line(
    daily_conversion,
    x="date",
    y="converted",
    color="group",
    title="Conversion Rate Over Time"
)

fig.show()

The graph presents the evolution of the daily conversion rate for the `control group` (old page) and the `treatment group` (new page with the recommendation system).

The visual analysis suggests that there is no consistent difference between the both versions over time. However, visual observation is not enough to confirm this hypothesis. For this reason, we must apply a Z-test to verify if the observed difference between the groups is statistically significant.

## **One-tailed hypothesis test**

In A/B testing of this project, we'll use a one-tailed test because the hypothesis we want to validate is directional. 

The objective of this case study is not simply verify if there is any difference between the two versions of the site. **What we specifically want to know is whether the new version of the site increases the conversion rate in relation to the old one.**

So, we define the hypotheses as follows:
- Null Hypothesis (H₀) : the conversion rate of the new page is less than or equal to the old one.
- Alternative Hypothesis (H₁): the conversion rate of the new page is higher than the old one.

*Observation: A two-tailed test would be used if we only wanted to verify if there is any difference between the versions, without assuming beforehand whether the new version would be better or worse.*


In [19]:
# Using only 5000 sample
control = sample_df[sample_df["group"] == "control"]
treatment = sample_df[sample_df["group"] == "treatment"]

conversions = [control["converted"].sum(), treatment["converted"].sum()]
n_obs = [len(control), len(treatment)]

z_stat, p_value = proportions_ztest(conversions, n_obs, alternative="larger")

print("Z-stat:", z_stat)
print("P-value:", p_value)

Z-stat: 0.4546639463663421
P-value: 0.324675511247162


Since the *p-value is high (32%)*, there is a relatively high probability that the observed difference occurred by chance, even if there is actually no improvement in conversion.

Based on this sample of 5,000 users, there is not enough statistical evidence to claim that the new version of the site increases the conversion rate compared to the old one.

In other words, the experiment does not demonstrate that the recommendation system improves the site's performance.

Let’s check if using the full dataframe yields a more significant p-value.

In [20]:
# All df
control = df[df["group"] == "control"]
treatment = df[df["group"] == "treatment"]

conversions = [control["converted"].sum(), treatment["converted"].sum()]
n_obs = [len(control), len(treatment)]

z_stat, p_value = proportions_ztest(conversions, n_obs, alternative="larger")

print("Z-stat:", z_stat)
print("P-value:", p_value)

Z-stat: 1.2942367891190543
P-value: 0.09779182143940718


After analyzing the test with 288,540 users, we observed that there is not enough statistical evidence to claim that the new version of the site increases the conversion rate compared to the old one. Although the treatment group shows a slightly higher conversion rate, the hypothesis test indicates that this difference is not statistically significant at the 5% significance level.

## Two-Tailed Hypothesis Test
To evaluate whether there is a statistically significant difference between the two versions of the site, we initially applied a two-tailed hypothesis test.

This type of test is appropriate when we do not assume the direction of the effect beforehand—meaning the new page could either increase or decrease the conversion rate."

In [21]:
# Using only 5000 sample
control = sample_df[sample_df["group"] == "control"]
treatment = sample_df[sample_df["group"] == "treatment"]

conversions = [control["converted"].sum(), treatment["converted"].sum()]
n_obs = [len(control), len(treatment)]

z_stat, p_value = proportions_ztest(conversions, n_obs)

print("Z-stat:", z_stat)
print("P-value:", p_value)

Z-stat: 0.4546639463663421
P-value: 0.649351022494324


The Z-statistic ≈ 0.45 indicates that the difference between the conversion rates of the two groups is very small. In statistical testing, Z-values close to 0 suggest that the groups are very similar. Furthermore, since the p-value is greater than 0.05, we fail to reject the null hypothesis. This means there is not enough statistical evidence to claim that the new version of the site has a conversion rate different from the current version.